# 21A — V5 E1 Source + Semantic QA Gate

This notebook assembles **DEV Expansion E1 Batch 01–08** and performs two pre-freeze checks:

1. source traceability,
2. content-only semantic consistency.

It does **not**:

- create positive/negative pairs,
- inspect 1–5 year gaps,
- rebalance chronology,
- generate astrology,
- score Control,
- research CONFIRM.

Rows are only **flagged** for review. Nothing is deleted or relabelled.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, re, time, unicodedata
import numpy as np
import pandas as pd

NOTEBOOK_VERSION="SAJU_ML_V5_E1_SOURCE_SEMANTIC_QA_20260817"

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repo.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

def norm(s):
    s=unicodedata.normalize("NFKD",str(s))
    s="".join(c for c in s if not unicodedata.combining(c))
    s=s.casefold()
    s=re.sub(r"\s+"," ",s)
    return s.strip()

ROOT=repo_root()
BASE=ROOT/"research/ml/artifacts/v5_dev_expansion_e1"
RES=BASE/"results"
OUT=ROOT/"research/ml/artifacts/v5_dev_expansion_e1_source_semantic_qa"
CORPUS=ROOT/"research/ml_corpus/v5_ground_truth"
OUT.mkdir(parents=True,exist_ok=True)

ROSTER=BASE/"V5_DEV_EXPANSION_E1_ROSTER_160.csv"
PROTO=CORPUS/"V5_E1_SOURCE_SEMANTIC_QA_PROTOCOL.json"

for p in [ROSTER,PROTO]:
    if not p.exists(): raise FileNotFoundError(p)

proto=json.load(open(PROTO,encoding="utf-8"))
assert proto["status"]=="PREDECLARED_BEFORE_E1_EVENT_FREEZE_AND_BEFORE_ANY_PAIR_GAP_INSPECTION"

paths=[]
for b in range(1,9):
    p=RES/f"batch_{b:02d}"/f"V5_DEV_EXPANSION_E1_BATCH_{b:02d}_EVENTS.csv"
    if not p.exists(): raise FileNotFoundError(p)
    paths.append((b,p))

roster=pd.read_csv(ROSTER)
assert len(roster)==160
assert roster.subject_id.nunique()==160

frames=[]
for b,p in paths:
    x=pd.read_csv(p)
    x["source_batch_file"]=str(p.relative_to(ROOT))
    x["event_row_in_batch"]=np.arange(1,len(x)+1)
    frames.append(x)

events=pd.concat(frames,ignore_index=True,sort=False)
assert events.subject_id.nunique()==160
assert set(events.subject_id)==set(roster.subject_id)

# Re-anchor immutable axis to roster and ensure no mismatch.
roster_axis=roster.set_index("subject_id")["preassigned_axis"]
assert events["preassigned_axis"].eq(events["subject_id"].map(roster_axis)).all()

def event_row_id(r):
    raw="|".join([
        "E1",
        str(r.get("subject_id","")),
        str(r.get("event_year","")),
        str(r.get("polarity","")),
        str(r.get("event_type","")),
        str(r.get("event_description","")),
        str(r.get("source_url",""))
    ])
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

events["event_row_id"]=events.apply(event_row_id,axis=1)
assert events.event_row_id.is_unique

assembled=OUT/"V5_E1_EVENTS_ASSEMBLED_PRE_QA.csv"
events.to_csv(assembled,index=False)

print("21A PREFLIGHT PASS")
print("E1 subjects:",events.subject_id.nunique())
print("event rows:",len(events))
print("eligible:",int((~events.exclude.astype(bool)).sum()))
print("excluded audit:",int(events.exclude.astype(bool).sum()))
print("No pair/year-gap computation performed.")


21A PREFLIGHT PASS
E1 subjects: 160
event rows: 333
eligible: 287
excluded audit: 46
No pair/year-gap computation performed.


## 1. Source traceability worklist

In [2]:

url_re=re.compile(r"^https?://",re.I)

keyword_families={
    "competitive":["title","champion","championship","gold","silver","bronze","medal","winner","won","final","cup","ranking","olympic","tournament"],
    "project":["award","prize","film","album","song","book","series","recording","released","festival","oscar","grammy","emmy","cannes","berlin","venice"],
    "status":["appointed","appointment","elected","election","chair","chairman","chief","ceo","minister","president","director","justice","judge","ambassador","leader","secretary","governor","mayor","resign","dismiss","removed"]
}

work=events.copy()
work["source_url_valid"]=work.source_url.fillna("").map(lambda x:bool(url_re.match(str(x).strip())))
work["axis_keyword_family"]=work.preassigned_axis.map({
    "COMPETITIVE":"competitive","PROJECT":"project","STATUS":"status"
})

work_path=OUT/"V5_E1_SOURCE_QA_WORKLIST.csv"
work.to_csv(work_path,index=False)

print("unique source URLs:",work.source_url.nunique())
print("invalid URL fields:",int((~work.source_url_valid).sum()))


unique source URLs: 216
invalid URL fields: 0


## 2. Fetch URLs conservatively

In [3]:

import requests
from bs4 import BeautifulSoup

CACHE=OUT/"http_cache"
CACHE.mkdir(parents=True,exist_ok=True)

session=requests.Session()
session.headers.update({
    "User-Agent":"Chartpalja-Saju-Research/1.0 source-semantic-audit",
    "Accept-Language":"en-US,en;q=0.8"
})

def cache_key(url):
    return hashlib.sha256(str(url).encode("utf-8")).hexdigest()

def fetch(url,timeout=25):
    key=cache_key(url)
    meta_p=CACHE/f"{key}.json"
    text_p=CACHE/f"{key}.txt"
    if meta_p.exists() and text_p.exists():
        return json.load(open(meta_p,encoding="utf-8")),text_p.read_text(encoding="utf-8",errors="ignore")

    meta={"url":url,"ok":False,"status_code":None,"final_url":"","title":"","error":""}
    text=""
    try:
        r=session.get(url,timeout=timeout,allow_redirects=True)
        meta["status_code"]=r.status_code
        meta["final_url"]=r.url
        if r.status_code<400:
            soup=BeautifulSoup(r.text,"html.parser")
            meta["title"]=soup.title.get_text(" ",strip=True) if soup.title else ""
            for tag in soup(["script","style","noscript","svg"]):
                tag.decompose()
            text=re.sub(r"\s+"," ",soup.get_text(" ",strip=True))
            meta["ok"]=len(text)>=80
        else:
            meta["error"]=f"HTTP_{r.status_code}"
    except Exception as e:
        meta["error"]=repr(e)[:500]

    json.dump(meta,open(meta_p,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
    text_p.write_text(text,encoding="utf-8")
    return meta,text

urls=sorted(set(work.loc[work.source_url_valid,"source_url"].astype(str)))
fetched={}
for i,url in enumerate(urls,1):
    fetched[url]=fetch(url)
    if i%25==0 or i==len(urls):
        print(f"fetched {i}/{len(urls)}")
    time.sleep(0.12)


fetched 25/216
fetched 50/216
fetched 75/216
fetched 100/216
fetched 125/216
fetched 150/216
fetched 175/216
fetched 200/216
fetched 216/216


## 3. Automatic source classification

In [4]:

def name_tokens(name):
    toks=[x for x in re.findall(r"[a-z0-9]+",norm(name)) if len(x)>=3]
    return toks[-1:] if toks else []

def source_check(r):
    if not r.source_url_valid:
        return {
            "source_verification_status":"INVALID_SOURCE_FIELD",
            "fetch_ok":False,"http_status":None,"page_title":"",
            "name_signal":False,"year_signal":False,"keyword_signal":False,
            "source_review_reason":"missing_or_malformed_http_url"
        }

    meta,text=fetched.get(str(r.source_url),({"ok":False},""))
    hay=norm((meta.get("title","") or "")+" "+text[:250000])

    nt=name_tokens(r["name"])
    name_signal=bool(nt) and all(t in hay for t in nt)

    year=""
    if pd.notna(r.event_year):
        try: year=str(int(float(r.event_year)))
        except: year=str(r.event_year)
    year_signal=bool(year and year in hay)

    fam=r.axis_keyword_family
    kws=keyword_families.get(fam,[])
    keyword_signal=any(norm(k) in hay for k in kws)

    fetch_ok=bool(meta.get("ok",False))
    auto=fetch_ok and name_signal and (year_signal or keyword_signal)

    reasons=[]
    if not fetch_ok: reasons.append("source_not_retrievable_or_text_too_short")
    if not name_signal: reasons.append("subject_name_not_detected")
    if not year_signal: reasons.append("event_year_not_detected")
    if not keyword_signal: reasons.append("axis_event_keyword_not_detected")

    return {
        "source_verification_status":"AUTO_PASS" if auto else "MANUAL_REVIEW",
        "fetch_ok":fetch_ok,
        "http_status":meta.get("status_code"),
        "page_title":meta.get("title",""),
        "name_signal":name_signal,
        "year_signal":year_signal,
        "keyword_signal":keyword_signal,
        "source_review_reason":"" if auto else ";".join(reasons)
    }

src_checks=pd.DataFrame([source_check(r) for _,r in work.iterrows()])
source_results=pd.concat([work.reset_index(drop=True),src_checks],axis=1)

source_results_path=OUT/"V5_E1_SOURCE_QA_AUTO_RESULTS.csv"
source_results.to_csv(source_results_path,index=False)

eligible_source_review=source_results[
    (~source_results.exclude.astype(bool))
    & (source_results.source_verification_status!="AUTO_PASS")
].copy()
eligible_source_review_path=OUT/"V5_E1_ELIGIBLE_SOURCE_MANUAL_REVIEW.csv"
eligible_source_review.to_csv(eligible_source_review_path,index=False)

print(source_results.source_verification_status.value_counts(dropna=False))
print("eligible source manual-review rows:",len(eligible_source_review))
print("unique URLs:",eligible_source_review.source_url.nunique())


AUTO_PASS        201
MANUAL_REVIEW    132
Name: source_verification_status, dtype: int64
eligible source manual-review rows: 99
unique URLs: 72


## 4. Predeclared semantic-consistency flags

In [5]:

def txt(r):
    return norm(" ".join([
        str(r.get("event_type","")),
        str(r.get("event_description","")),
        str(r.get("eligibility_note","")),
        str(r.get("exclude_reason",""))
    ]))

def semantic_reasons(r):
    reasons=[]
    t=txt(r)
    q=norm(r.get("source_quality",""))

    if q in {"medium","low"}:
        reasons.append("SOURCE_QUALITY_MEDIUM_OR_LOW")

    if any(k in t for k in ["semantic", "recheck", "verify exact", "verify ", "ambiguous"]):
        reasons.append("EXPLICIT_FIRST_PASS_SEMANTIC_NOTE")

    if not bool(r["exclude"]):
        if r.preassigned_axis=="COMPETITIVE" and r.polarity=="negative":
            if any(k in t for k in ["draw","tie","tied"]):
                reasons.append("COMPETITIVE_NEGATIVE_DRAW_OR_TIE_MECHANISM")

        if r.preassigned_axis=="STATUS" and r.polarity=="negative":
            if any(k in t for k in ["resign","resignation","removed","removal","collapse","disqualif","government_end","government end","lost office"]):
                reasons.append("STATUS_NEGATIVE_MECHANISM_POLICY_CHECK")

        if r.preassigned_axis=="PROJECT" and r.polarity=="positive":
            if any(k in t for k in ["lifetime","honorary","career award","grand prix"]):
                reasons.append("PROJECT_DISCRETE_OUTCOME_POLICY_CHECK")

    return reasons

sem=events.copy()
sem["semantic_review_reasons"]=sem.apply(lambda r:";".join(semantic_reasons(r)),axis=1)
sem["semantic_review_flag"]=sem.semantic_review_reasons.ne("")

# Same subject-year opposite-polarity consistency check.
elig=sem[~sem.exclude.astype(bool)].copy()
yrpol=(
    elig.groupby(["subject_id","event_year"]).polarity
    .agg(lambda x:set(x))
)
collision_keys=set(
    idx for idx,s in yrpol.items()
    if {"positive","negative"}.issubset(s)
)
if collision_keys:
    mask=sem.apply(lambda r:(r.subject_id,r.event_year) in collision_keys,axis=1)
    sem.loc[mask,"semantic_review_reasons"]=sem.loc[mask,"semantic_review_reasons"].map(
        lambda x:(x+";" if x else "")+"SAME_SUBJECT_YEAR_OPPOSITE_POLARITY_CONSISTENCY_CHECK"
    )
    sem.loc[mask,"semantic_review_flag"]=True

semantic_review=sem[
    (~sem.exclude.astype(bool))
    & sem.semantic_review_flag
].copy()

semantic_review_path=OUT/"V5_E1_ELIGIBLE_SEMANTIC_MANUAL_REVIEW.csv"
semantic_review.to_csv(semantic_review_path,index=False)

collision_rows=sem[
    sem.semantic_review_reasons.str.contains(
        "SAME_SUBJECT_YEAR_OPPOSITE_POLARITY_CONSISTENCY_CHECK",
        na=False
    )
].copy()
collision_path=OUT/"V5_E1_SAME_YEAR_OPPOSITE_POLARITY_REVIEW.csv"
collision_rows.to_csv(collision_path,index=False)

print("eligible semantic-review rows:",len(semantic_review))
print("same-year opposite-polarity subject-years:",len(collision_keys))
print("No pair gap or local-pair generation was performed.")


eligible semantic-review rows: 98
same-year opposite-polarity subject-years: 9
No pair gap or local-pair generation was performed.


## 5. Freeze the QA decision, not the event corpus

In [6]:

eligible=events[~events.exclude.astype(bool)]

source_pending=len(eligible_source_review)
semantic_pending=len(semantic_review)

status=(
    "V5_E1_SOURCE_SEMANTIC_MANUAL_BACKFILL_REQUIRED"
    if (source_pending>0 or semantic_pending>0)
    else
    "V5_E1_SOURCE_SEMANTIC_AUTO_GATE_PASS_READY_FOR_EVENT_FREEZE"
)

summary={
    "version":"V5_E1_SOURCE_SEMANTIC_QA_SUMMARY_V1",
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":status,
    "E1_subjects_n":int(events.subject_id.nunique()),
    "event_rows_n":int(len(events)),
    "eligible_rows_n":int(len(eligible)),
    "excluded_rows_n":int(events.exclude.astype(bool).sum()),
    "unique_source_urls_n":int(events.source_url.nunique()),
    "eligible_source_manual_review_rows_n":int(source_pending),
    "eligible_source_manual_review_unique_urls_n":int(eligible_source_review.source_url.nunique()),
    "eligible_semantic_manual_review_rows_n":int(semantic_pending),
    "eligible_semantic_manual_review_subjects_n":int(semantic_review.subject_id.nunique()),
    "same_year_opposite_polarity_subject_years_n":int(len(collision_keys)),
    "rules":{
        "event_rows_mutated":False,
        "membership_changed":False,
        "pair_gap_inspected":False,
        "pairs_generated":False,
        "chronology_balanced":False,
        "astrology_generated":False,
        "control_scored":False,
        "confirm_researched":False
    },
    "next_rule":(
        "Send the eligible source and semantic manual-review CSVs for independent verification. "
        "Do not freeze E1 events until all review items are resolved by immutable override."
        if status=="V5_E1_SOURCE_SEMANTIC_MANUAL_BACKFILL_REQUIRED"
        else
        "Proceed to E1 final event freeze."
    )
}
summary_path=OUT/"V5_E1_SOURCE_SEMANTIC_QA_SUMMARY.json"
json.dump(summary,open(summary_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

decision={
    "version":"V5_E1_SOURCE_SEMANTIC_QA_DECISION_V1",
    "status":status,
    "assembled_events_sha256":sha256_file(assembled),
    "source_auto_results_sha256":sha256_file(source_results_path),
    "eligible_source_manual_review_sha256":sha256_file(eligible_source_review_path),
    "eligible_semantic_manual_review_sha256":sha256_file(semantic_review_path),
    "same_year_collision_review_sha256":sha256_file(collision_path),
    "summary_sha256":sha256_file(summary_path),
    "event_freeze_allowed":status=="V5_E1_SOURCE_SEMANTIC_AUTO_GATE_PASS_READY_FOR_EVENT_FREEZE",
    "pairing_allowed":False,
    "astrology_generation_allowed":False,
    "confirm_event_research_allowed":False
}
decision_path=OUT/"V5_E1_SOURCE_SEMANTIC_QA_DECISION.json"
json.dump(decision,open(decision_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "version": "V5_E1_SOURCE_SEMANTIC_QA_SUMMARY_V1",
  "created_at": "2026-08-17T05:41:16",
  "status": "V5_E1_SOURCE_SEMANTIC_MANUAL_BACKFILL_REQUIRED",
  "E1_subjects_n": 160,
  "event_rows_n": 333,
  "eligible_rows_n": 287,
  "excluded_rows_n": 46,
  "unique_source_urls_n": 216,
  "eligible_source_manual_review_rows_n": 99,
  "eligible_source_manual_review_unique_urls_n": 72,
  "eligible_semantic_manual_review_rows_n": 98,
  "eligible_semantic_manual_review_subjects_n": 66,
  "same_year_opposite_polarity_subject_years_n": 9,
  "rules": {
    "event_rows_mutated": false,
    "membership_changed": false,
    "pair_gap_inspected": false,
    "pairs_generated": false,
    "chronology_balanced": false,
    "astrology_generated": false,
    "control_scored": false,
    "confirm_researched": false
  },
  "next_rule": "Send the eligible source and semantic manual-review CSVs for independent verification. Do not freeze E1 events until all review items are resolved by immutable override."
}


## Return to ChatGPT

After `Kernel Restart -> Run All`, send:

```text
V5_E1_SOURCE_SEMANTIC_QA_DECISION.json
V5_E1_SOURCE_SEMANTIC_QA_SUMMARY.json
V5_E1_ELIGIBLE_SOURCE_MANUAL_REVIEW.csv
V5_E1_ELIGIBLE_SEMANTIC_MANUAL_REVIEW.csv
V5_E1_SAME_YEAR_OPPOSITE_POLARITY_REVIEW.csv
```

Do not edit the review CSVs.

I can then resolve all remaining review items in one pass, create immutable overrides, and build the E1 final event-freeze notebook.
